In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider


In [2]:
# Длины сегментов
L_hip = 10  # длина бедра
L_leg = 10  # длина голени
L_caromyslo = 2  # длина каромысла голени
L_tyaga = 10  # длина тяги голени
L_richag = 2.1  # длина рычага двигателя 3



In [3]:
# Решение уравнений обратной кинематики
def inverse_kinematics(target_x, target_y, L_hip, L_leg):
    # Угол голени
    cos_angle_leg = (target_x ** 2 + target_y ** 2 - L_hip ** 2 - L_leg ** 2) / (2 * L_hip * L_leg)
    sin_angle_leg = np.sqrt(1 - cos_angle_leg ** 2)
    theta_leg = np.arctan2(sin_angle_leg, cos_angle_leg)

    # Угол бедра
    k1 = L_hip + L_leg * cos_angle_leg
    k2 = L_leg * sin_angle_leg
    theta_hip = np.arctan2(target_y, target_x) - np.arctan2(k2, k1)

    # Угол рычага
    theta_richag = np.pi / 3  # начальное значение угла рычага двигателя 3

    return theta_hip, theta_leg, theta_richag

In [4]:
# Вычисление положений точек
def calculate_positions(theta_hip, theta_leg, theta_richag, L_hip, L_leg, L_caromyslo, L_tyaga, L_richag):
    # Точка конца бедра (A)
    A_x = L_hip * np.cos(theta_hip)
    A_y = L_hip * np.sin(theta_hip)

    # Точка крепления бедра и голени (B)
    B_x = A_x
    B_y = A_y

    # Точка конца голени (ступня) (E)
    E_x = B_x + L_leg * np.cos(theta_hip + theta_leg)
    E_y = B_y + L_leg * np.sin(theta_hip + theta_leg)

    # Точка конца каромысла (B_ext)
    B_ext_x = B_x - L_caromyslo * np.cos(theta_hip + theta_leg)
    B_ext_y = B_y - L_caromyslo * np.sin(theta_hip + theta_leg)

    # Точка конца тяги (C)
    C_x = B_ext_x + L_tyaga * np.cos(theta_richag)
    C_y = B_ext_y + L_tyaga * np.sin(theta_richag)

    # Точка конца рычага двигателя 3 (D)
    D_x = L_richag * np.cos(theta_richag)
    D_y = L_richag * np.sin(theta_richag)

    # Обновляем конечную точку тяги, соединяя её с концом рычага
    C_x = D_x
    C_y = D_y

    return (0, 0), (A_x, A_y), (B_x, B_y), (B_ext_x, B_ext_y), (C_x, C_y), (D_x, D_y), (E_x, E_y)


In [5]:
# Визуализация системы рычагов
def plot_lever_system(target_x, target_y):
    theta_hip, theta_leg, theta_richag = inverse_kinematics(target_x, target_y, L_hip, L_leg)
    origin, A, B, B_ext, C, D, E = calculate_positions(theta_hip, theta_leg, theta_richag, L_hip, L_leg, L_caromyslo,
                                                       L_tyaga, L_richag)

    # Вывод углов в консоль
    print(f"Угол бедра (Двигатель 2): {np.degrees(theta_hip):.2f} градусов")
    print(f"Угол рычага двигателя 3: {np.degrees(theta_richag):.2f} градусов")

    plt.figure()
    plt.plot([origin[0], A[0]], [origin[1], A[1]], '-o', color='blue', label='Бедро')
    plt.plot([A[0], E[0]], [A[1], E[1]], '-o', color='green', label='Голень')
    plt.plot([B[0], B_ext[0]], [B[1], B_ext[1]], '-o', color='green', label='Каромысло голени')
    plt.plot([B_ext[0], C[0]], [B_ext[1], C[1]], '-o', color='purple', label='Длинная тяга голени')
    plt.plot([origin[0], D[0]], [origin[1], D[1]], '-o', color='brown', label='Рычаг двигателя 3')
    plt.plot(E[0], E[1], 'go', label='Ступня')
    plt.plot(target_x, target_y, 'ro', label='Целевая точка (ступня)')
    plt.xlim(-L_hip - L_leg - L_caromyslo, L_hip + L_leg + L_caromyslo)
    plt.ylim(-L_hip - L_leg - L_caromyslo, L_hip + L_leg + L_caromyslo)
    plt.gca().set_aspect('equal')
    plt.legend()
    plt.show()

In [6]:
# Создание ползунков для динамического изменения координат целевой точки
interact(plot_lever_system, target_x=FloatSlider(min=-20, max=20, step=0.1, value=0),
         target_y=FloatSlider(min=-20, max=20, step=0.1, value=-10))

interactive(children=(FloatSlider(value=0.0, description='target_x', max=20.0, min=-20.0), FloatSlider(value=-…

<function __main__.plot_lever_system(target_x, target_y)>

In [7]:
# import numpy as np
# import matplotlib.pyplot as plt
# from ipywidgets import interact, FloatSlider

# # Длины сегментов
# L_hip = 10       # длина бедра
# L_leg = 10       # длина голени
# L_caromyslo = 2  # длина каромысла голени
# L_tyaga = 10     # длина тяги голени
# L_richag = 2.1   # длина рычага двигателя 3

# # Смещение оси двигателя 3
# offset_3_x = 1
# offset_3_y = -1

# # Ограничения по углам (в радианах)
# theta_hip_min = np.deg2rad(135)
# theta_hip_max = np.deg2rad(270)
# theta_richag_min = np.deg2rad(-45)
# theta_richag_max = np.deg2rad(135)

# # Функция для проверки, находится ли угол в заданном секторе
# def in_sector(angle, min_angle, max_angle):
#     angle = angle % (2 * np.pi)
#     if min_angle < max_angle:
#         return min_angle <= angle <= max_angle
#     else:
#         return min_angle <= angle or angle <= max_angle

# # Функция для приведения угла в заданный сектор
# def adjust_angle_to_sector(angle, min_angle, max_angle):
#     angle = angle % (2 * np.pi)
#     if in_sector(angle, min_angle, max_angle):
#         return angle
#     else:
#         delta = (max_angle - min_angle) % (2 * np.pi)
#         if angle < min_angle:
#             angle += delta
#         elif angle > max_angle:
#             angle -= delta
#         return angle

# # Решение уравнений обратной кинематики с учетом ограничений
# def inverse_kinematics(target_x, target_y, L_hip, L_leg):
#     # Угол голени
#     cos_angle_leg = (target_x**2 + target_y**2 - L_hip**2 - L_leg**2) / (2 * L_hip * L_leg)
#     sin_angle_leg = np.sqrt(1 - cos_angle_leg**2)
#     theta_leg = np.arctan2(sin_angle_leg, cos_angle_leg)
    
#     # Угол бедра с учетом ограничений
#     k1 = L_hip + L_leg * cos_angle_leg
#     k2 = L_leg * sin_angle_leg
#     theta_hip = np.arctan2(target_y, target_x) - np.arctan2(k2, k1)
#     theta_hip = adjust_angle_to_sector(theta_hip, theta_hip_min, theta_hip_max)
    
#     # Угол рычага с учетом ограничений
#     theta_richag = np.arctan2(target_y - offset_3_y, target_x - offset_3_x)
#     theta_richag = adjust_angle_to_sector(theta_richag, theta_richag_min, theta_richag_max)
    
#     return theta_hip, theta_leg, theta_richag

# # Вычисление положений точек
# def calculate_positions(theta_hip, theta_leg, theta_richag, L_hip, L_leg, L_caromyslo, L_tyaga, L_richag, offset_3_x, offset_3_y):
#     # Точка конца бедра (A)
#     A_x = L_hip * np.cos(theta_hip)
#     A_y = L_hip * np.sin(theta_hip)
    
#     # Точка крепления бедра и голени (B)
#     B_x = A_x
#     B_y = A_y
    
#     # Точка конца голени (ступня) (E)
#     E_x = B_x + L_leg * np.cos(theta_hip + theta_leg)
#     E_y = B_y + L_leg * np.sin(theta_hip + theta_leg)
    
#     # Точка конца каромысла (B_ext) 
#     B_ext_x = B_x - L_caromyslo * np.cos(theta_hip + theta_leg)
#     B_ext_y = B_y - L_caromyslo * np.sin(theta_hip + theta_leg)
    
#     # Точка конца тяги (C)
#     C_x = B_ext_x + L_tyaga * np.cos(theta_richag)
#     C_y = B_ext_y + L_tyaga * np.sin(theta_richag)
    
#     # Точка конца рычага двигателя 3 (D) с учетом смещения
#     D_x = offset_3_x + L_richag * np.cos(theta_richag)
#     D_y = offset_3_y + L_richag * np.sin(theta_richag)
    
#     # Обновляем конечную точку тяги, соединяя её с концом рычага
#     C_x = D_x
#     C_y = D_y
    
#     return (0, 0), (A_x, A_y), (B_x, B_y), (B_ext_x, B_ext_y), (C_x, C_y), (D_x, D_y), (E_x, E_y)

# # Визуализация системы рычагов
# def plot_lever_system(target_x, target_y):
#     theta_hip, theta_leg, theta_richag = inverse_kinematics(target_x, target_y, L_hip, L_leg)

#     if not in_sector(theta_hip, theta_hip_min, theta_hip_max):
#         print("Недопустимое решение: Угол бедра выходит за пределы ограничений.")
#         return
    
#     if not in_sector(theta_richag, theta_richag_min, theta_richag_max):
#         print("Недопустимое решение: Угол рычага выходит за пределы ограничений.")
#         return
    
#     origin, A, B, B_ext, C, D, E = calculate_positions(theta_hip, theta_leg, theta_richag, L_hip, L_leg, L_caromyslo, L_tyaga, L_richag, offset_3_x, offset_3_y)
    
#     # Вывод углов в консоль
#     print(f"Угол бедра (Двигатель 2): {np.degrees(theta_hip):.2f} градусов")
#     print(f"Угол рычага (Двигатель 3): {np.degrees(theta_richag):.2f} градусов")

#     plt.figure()
#     plt.plot([origin[0], A[0]], [origin[1], A[1]], '-o', color='blue', label='Бедро')
#     plt.plot([A[0], E[0]], [A[1], E[1]], '-o', color='green', label='Голень')
#     plt.plot([B[0], B_ext[0]], [B[1], B_ext[1]], '-o', color='green', label='Каромысло голени')
#     plt.plot([B_ext[0], C[0]], [B_ext[1], C[1]], '-o', color='purple', label='Длинная тяга голени')
#     plt.plot([offset_3_x, D[0]], [offset_3_y, D[1]], '-o', color='brown', label='Рычаг двигателя 3')
#     plt.plot(E[0], E[1], 'go', label='Ступня')
#     plt.plot(target_x, target_y, 'ro', label='Целевая точка (ступня)')
#     plt.xlim(-15, 15)
#     plt.ylim(-15, 15)
#     plt.gca().set_aspect('equal')
#     plt.legend()
#     plt.show()

# # Создание ползунков для динамического изменения координат целевой точки
# interact(plot_lever_system, target_x=FloatSlider(min=-15, max=15, step=0.1, value=0), target_y=FloatSlider(min=-15, max=15, step=0.1, value=-10))


In [8]:
# Сейчас оно работает немного неправильно и не все длины узлов соблюдаются. Давай я попробую объяснить тебе расчёт кинематики, а ты уже перестроишь данный код. Есть фиксированные точки, которые мы знаем. Это точка 2 и 3 двигателя (2ой двигатель в 0,0 - 3ий двигатель со смещением указанным в переменных). Также мы имеем различные длины (или углы). Всё остальное нам неизвестно изначально.

# Требуется составить такую функцию, которая на вход получает координаты ступни, а выдаёт углы поворота двигателей (к которым фактически закреплены некоторые узлы, а именно за 2 двигателем - бедро, а за 3ьим - рычаг двигателя).
# При этом у такой функции будет система ограничений, ибо в ходе расчётов окажется, что система может иметь ряд решений (но нам нужно одно, если это возможно)

# И так, есть ступня в коорндинатах x,y. Для расчёта угла бедра этого уже достаточно - требуется построить в точке 0,0 окружность, которая описывает все положения конца бедра и найти пересечения с другой окружностью, центр которой x,y. Радиусы этих окружностей это длины голени и бедра соответственно. В дальнейшем каждый последующую связку будем представлять именно как поиск пересечений. Но пересечений может не быть, может быть много и так далее. 1 случай - пересечений нет, тогда смело возвращаем ошибку о невозможности найти решение. 2 случай - пересечение одно или их бесконечно много - это краевые случаи, когда либо ещё чуть чуть и узел не сможет дотянуться до другого, либо он уже находится на тех же координатах, что и другой. Этот вариант ещё можно считать подходящим, но стоит отдельно обрабатывать, чтобы далее подавать "предупреждение". 3и случай - пересечений два. Нужно лишь выбрать нужное. Именно про эти ограничения я говорил, и в дальшнейшем я опишу какое правилило применять к каждой паре узлов. Если построить общую хорду между точками пересечений (когда их две) - можно определять по длине хорды, как близко мы к крайним положениям (запомни это, ибо я бы это значение также расчитывал для анализа, когда будем заниматься обучением робота в хожждении).
# Так вот, при 3 случае - у нас есть 2 точки пересечения окружностей голени и бедра. Какое выбрать? Так как сейчас в коде нога смотри как бы "лицом" по оси OX (то есть робот будет ходить вперёд в стороноу увеличения значения X), то нужно выбирать точку с меньшей координатой X (то есть против направления ноги). Когда точек бесконечно - пусть выдаётся ошибка. Если точка одна - выдаёшь её. Если точек нет - ошибку об отсутствия пересечений. И так, выбрав нужную точку пересечения можно уже построить два отрезка - голень и бедро. А бедро даст ответ на угол одного из двигателей (2 двигатель).

# Следующим узлом будет каромысло голени и сторона качалки A. Мы знаем, что качалка начинается в точке 0,0 (где и бедро). Поэтому и стороны A и B качалки начинаются оттуда. Угол между ними мы также знаем. Можно построить полноценный треугольник. Но мы не знаем угол наклона стороны A (соответственно всего треугольника, который олицетворяет качалку). Онднако мы уже знаем положение голени и каромысла (каромысло это отрезок, конец конторого находится в точке пересечения голени и бедра, а другой его конец просто сдвинут (угол и длина сдива указана как переменные)), а между ними находится длинная тяга голени определённой длины. Построив окружности от 0,0 (с радиусом в длину стороны A) и от конца каромысла (с радиусом в длину длинной тяги голени) мы снова получим задачу нахождения пересечений, которая даст ответ в какой точке должна находится вторая точка отрезка стороны A (первая точка, как мы уже говорили исходит из 0,0 - начала бедра). Далее снова 3 случая пересечений, про правила работы с которыми я уже писал ранее. Однако, что делать при двух пересечениях? Тут правильно такое: берём точку которая выше (то есть больше значение Y). и таким образом мы находим положение стороны А качалки.

# Теперь зная форму качалки можнжо построить все 3 стороны качалки. Остаётся последнее сочление. Двигатель 3 вращает рычаг двигателя 3 (про сдвиг относительно начала координат я уже писал). От оси двигателя 3 уже можно построить окружность (с радиусом рычага двигателя 3), а от точки пересечения стороны B и стороны C (один из углов треугольника качалки) строим ещё одну окружность (с радиусом в длину малая тяги). Так мы снова получим задачу пересечения окружностей. При выборе из двух точек - выбираем ту, которая идёт в сторону направления оси OX (в нашем случае, выходит, имеет больше значение X). Так мы находим последнее пересечение узлов.

# Теперь зная все точки пересечения мы можем ПРАВИЛЬНО построить график, который симулирует связь физических сочленений робота, а главное расчитать по входным X,Y - углы двигателя 2 и 3. Сможешь дорабоатть код, который я выше приложил с такой логикой расчётов? До этого длинная тяга голени почему-то не имела заданной длины. Советую проверить есть ли специальная функция, которая рассчитывает пересечение двух окружностей, чтобы уже потом обрабатывать все случаи и при случае с 2 точками - выбирать нужное, согласно описанным моим условиям. Если будут вопросы - обращайся. Как изображение - приложил как выглядит график (но работающий немного неправильно).

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from leg import *

# Функция для визуализации
def plot_lever_system(target_x, target_y):
    result = inverse_kinematics(target_x, target_y)
    if isinstance(result, str):
        print(result)
        return
    
    # Извлекаем углы и позиции
    motor2_angle = result["motor2_angle"]
    motor3_angle = result["motor3_angle"]
    positions = result["positions"]
    origin = positions["origin"]
    knee = positions["knee"]
    B_ext = positions["B_ext"]
    D = positions["D"]
    P = positions["P"]
    R = positions["R"]
    foot = positions["foot"]
    
    print(f"Угол поворота для двигателя 2: {motor2_angle:.2f} градусов")
    print(f"Угол поворота для двигателя 3: {motor3_angle:.2f} градусов")
    
    # Построение графика
    plt.figure()
    plt.plot([origin[0], knee[0]], [origin[1], knee[1]], '-o', color='blue', label='Бедро')
    plt.plot([knee[0], foot[0]], [knee[1], foot[1]], '-o', color='green', label='Голень')
    plt.plot([knee[0], B_ext[0]], [knee[1], B_ext[1]], '-o', color='green', label='Каромысло голени')
    plt.plot([B_ext[0], D[0]], [B_ext[1], D[1]], '-o', color='purple', label='Длинная тяга голени')
    plt.plot([0, D[0]], [0, D[1]], '-o', color='orange', label='Сторона A качалки')
    plt.plot([0, P[0]], [0, P[1]], '-o', color='orange', label='Сторона B качалки')
    plt.plot([D[0], P[0]], [D[1], P[1]], '-o', color='orange', label='Сторона C качалки')
    plt.plot([P[0], R[0]], [P[1], R[1]], '-o', color='cyan', label='Малая тяга')
    plt.plot([offset_3_x, R[0]], [offset_3_y, R[1]], '-o', color='red', label='Рычаг двигателя 3')
    plt.plot(target_x, target_y, 'ro', label='Ступня (целевая точка)')
    
    plt.xlim(-20, 20)
    plt.ylim(-20, 20)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.grid(color='gray', linestyle='--', linewidth=0.5)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend(fontsize=8)
    plt.show()

# Интерактивная визуализация
interact(
    plot_lever_system,
    target_x=FloatSlider(min=-15, max=15, step=0.1, value=0),
    target_y=FloatSlider(min=-15, max=5, step=0.1, value=-12)
)

interactive(children=(FloatSlider(value=0.0, description='target_x', max=15.0, min=-15.0), FloatSlider(value=-…

<function __main__.plot_lever_system(target_x, target_y)>